In [32]:
import pandas as pd
import numpy as np

# Obtener la data
data = pd.read_csv('../data/ProcessedData-2023-to-2025.csv', parse_dates=['dia'])
data = data.dropna()

INCLUDE = ['precio', 'precio_z', 'dia']
EXCLUDE = ['precio_lag_1h', 'precio_lag_24h', 'precio_lag_168h', 'precio_z_lag_1h', 'precio_z_lag_24h', 'precio_z_lag_168h', 'dia_anual_sin', 'dia_anual_cos']
FEATURES = list(set(data.columns) - set(INCLUDE + EXCLUDE))

TARGET = 'precio'

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   dia            1096 non-null   datetime64[us]
 1   precio         1096 non-null   float64       
 2   precio_z       1096 non-null   float64       
 3   dia_semanal_1  1096 non-null   int64         
 4   dia_semanal_2  1096 non-null   int64         
 5   dia_semanal_3  1096 non-null   int64         
 6   dia_semanal_4  1096 non-null   int64         
 7   dia_semanal_5  1096 non-null   int64         
 8   dia_semanal_6  1096 non-null   int64         
 9   dia_anual_sin  1096 non-null   float64       
 10  dia_anual      1096 non-null   int64         
 11  year           1096 non-null   int64         
 12  year_2024      1096 non-null   int64         
 13  year_2025      1096 non-null   int64         
dtypes: datetime64[us](1), float64(3), int64(10)
memory usage: 120.0 KB


In [33]:
# # Sort chronologically first
# data = data.sort_values(['dia']).reset_index(drop=True)
#
# X = data[FEATURES]
# y = data['precio']
#
# # Split at mid-2025
# split_date = pd.Timestamp('2025-07-01')
#
# train_mask = data['dia'] < split_date
# test_mask  = data['dia'] >= split_date
#
# X_train, y_train = X[train_mask], y[train_mask]
# X_test,  y_test  = X[test_mask],  y[test_mask]
#
# print(f"Train: {len(X_train)} rows ({data.loc[train_mask, 'dia'].dt.year.unique().tolist()})")
# print(f"Test:  {len(X_test)} rows  ({data.loc[test_mask,  'dia'].dt.year.unique().tolist()})")

In [34]:
# Sort chronologically first
data = data.sort_values(['dia']).reset_index(drop=True)

X = data[FEATURES]

TARGET_1LP = f'{TARGET}_1lp'

# Log-transform the target
data[TARGET_1LP] = np.log1p(data[TARGET])
data[TARGET] = data.groupby('year')[TARGET_1LP].transform(
    lambda x: (x - x.mean()) / x.std()
)

y = data[TARGET]

# Split at mid-2025
split_date = pd.Timestamp('2025-07-01')

train_mask = data['dia'] < split_date
test_mask  = data['dia'] >= split_date

# Drop NaNs and align X/y
X_train = X[train_mask].dropna()
y_train = y[train_mask].loc[X_train.index]

X_test  = X[test_mask].dropna()
y_test  = y[test_mask].loc[X_test.index]

year_stats = data.loc[train_mask].groupby('year')[TARGET].agg(['mean', 'std'])

print(f"Train: {len(X_train)} rows ({data.loc[train_mask, 'dia'].dt.year.unique().tolist()})")
print(f"Test:  {len(X_test)} rows  ({data.loc[test_mask,  'dia'].dt.year.unique().tolist()})")
print(f"NaNs dropped — Train: {train_mask.sum() - len(X_train)}, Test: {test_mask.sum() - len(X_test)}")

y_train.info()

Train: 912 rows ([2023, 2024, 2025])
Test:  184 rows  ([2025])
NaNs dropped — Train: 0, Test: 0
<class 'pandas.Series'>
RangeIndex: 912 entries, 0 to 911
Series name: precio
Non-Null Count  Dtype  
--------------  -----  
912 non-null    float64
dtypes: float64(1)
memory usage: 7.3 KB


In [35]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error

# TimeSeriesSplit is critical — never use random KFold on time series data
# it would leak future data into training folds
tscv = TimeSeriesSplit(n_splits=2)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    train_dates = data.loc[X_train.iloc[train_idx].index, 'dia']
    val_dates   = data.loc[X_train.iloc[val_idx].index, 'dia']
    print(f"Fold {fold}: Train [{train_dates.min().date()} → {train_dates.max().date()}] "
          f"Val [{val_dates.min().date()} → {val_dates.max().date()}]")

pipelines = {
    'Ridge': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=True, interaction_only=False, order='C')),
        ('scaler', StandardScaler()),
        ('model', Ridge())
    ]),
    'Lasso': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=True, interaction_only=False, order='C')),
        ('scaler', StandardScaler()),
        ('model', Lasso(max_iter=100000))
    ]),
    'ElasticNet': Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNet(max_iter=100000))
    ]),
    'RandomForest': Pipeline([
        ('model', RandomForestRegressor(n_jobs=-1, random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('model', GradientBoostingRegressor(random_state=42))
    ]),
}

param_grids = {
    'Ridge': {
        'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000]
    },
    'Lasso': {
        'model__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]
    },
    'ElasticNet': {
        'model__alpha':   [0.001, 0.01, 0.1, 1, 10, 100],
        'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
    },
    'RandomForest': {
        'model__n_estimators': [50, 100, 300],
        'model__max_depth':    [None, 10, 20, 50],
        'model__min_samples_leaf': [1, 5, 10, 25],
    },
    'GradientBoosting': {
        'model__n_estimators':  [50, 100, 300],
        'model__max_depth':     [3, 5, 7, 12],
        'model__learning_rate': [0.001, 0.01, 0.1, 0.2, 0.3],
    },
}

results = {}
for name, pipeline in pipelines.items():
    print(f"\nFitting {name}...")
    grid = GridSearchCV(
        pipeline,
        param_grids[name],
        # cv=[(np.arange(len(X_train)), np.arange(len(X_train)))],
        cv=tscv,
        scoring='r2',
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train, y_train)

    # y_pred = grid.predict(X_test)
    # test_r2  = r2_score(y_test, y_pred)
    # test_mae = mean_absolute_error(y_test, y_pred)

    y_pred_log  = grid.predict(X_test)
    y_pred_z    = np.expm1(y_pred_log)

    # Inverse z-transform per year using training stats
    y_test_dates = data.loc[X_test.index, ['dia', 'year']]
    y_pred_orig  = pd.Series(index=X_test.index, dtype=float)
    y_test_orig  = pd.Series(index=X_test.index, dtype=float)

    for year, group in y_test_dates.groupby('year'):
        mean = year_stats.loc[year, 'mean']
        std  = year_stats.loc[year, 'std']
        y_pred_orig[group.index] = y_pred_z[group.index - X_test.index[0]] * std + mean
        y_test_orig[group.index] = data.loc[group.index, TARGET]

    test_r2  = r2_score(y_test_orig, y_pred_orig)
    test_mae = mean_absolute_error(y_test_orig, y_pred_orig)

    results[name] = {
        'best_params': grid.best_params_,
        'cv_r2':       grid.best_score_,
        'test_r2':     test_r2,
        'test_mae':    test_mae,
        'model':       grid.best_estimator_
    }

    print(f"  Best params : {grid.best_params_}")
    print(f"  CV R²       : {grid.best_score_:.4f}")
    print(f"  Test R²     : {test_r2:.4f}")
    print(f"  Test MAE    : {test_mae:.4f}")

# Summary table
print("\n=== RESUMEN ===")
print(f"{'Model':<20} {'CV R²':>8} {'Test R²':>10} {'Test MAE':>10}")
print("-" * 50)
for name, r in sorted(results.items(), key=lambda x: x[1]['test_r2'], reverse=True):
    print(f"{name:<20} {r['cv_r2']:>8.4f} {r['test_r2']:>10.4f} {r['test_mae']:>10.4f}")

Fold 0: Train [2023-01-01 → 2023-10-31] Val [2023-11-01 → 2024-08-30]
Fold 1: Train [2023-01-01 → 2024-08-30] Val [2024-08-31 → 2025-06-30]

Fitting Ridge...
Fitting 2 folds for each of 9 candidates, totalling 18 fits
  Best params : {'model__alpha': 10}
  CV R²       : 0.2075
  Test R²     : -1.3004
  Test MAE    : 1.0700

Fitting Lasso...
Fitting 2 folds for each of 8 candidates, totalling 16 fits
  Best params : {'model__alpha': 0.1}
  CV R²       : 0.0450
  Test R²     : 0.1129
  Test MAE    : 0.5816

Fitting ElasticNet...
Fitting 2 folds for each of 30 candidates, totalling 60 fits
  Best params : {'model__alpha': 0.1, 'model__l1_ratio': 0.9}
  CV R²       : -0.0314
  Test R²     : 0.0584
  Test MAE    : 0.5855

Fitting RandomForest...
Fitting 2 folds for each of 48 candidates, totalling 96 fits
  Best params : {'model__max_depth': 10, 'model__min_samples_leaf': 25, 'model__n_estimators': 100}
  CV R²       : 0.2734
  Test R²     : 0.0255
  Test MAE    : 0.6268

Fitting GradientBo